#### Mục đích & Quy mô: Cung cấp thông tin dinh dưỡng chi tiết của 77 loại ngũ cốc ăn sáng từ nhiều nhà sản xuất lớn (Kellogg's, General Mills, Post, Quaker Oats,...).

#### Các trường dữ liệu (16 columns):

name: Tên sản phẩm ngũ cốc.

mfr: Mã nhà sản xuất (A, G, K, N, P, Q, R).

type: Loại ngũ cốc (C = Cold/Lạnh, H = Hot/Nóng).

calories: Lượng calo mỗi khẩu phần.

protein: Lượng chất đạm (g).

fat: Lượng chất béo (g).

sodium: Hàm lượng natri (mg).

fiber: Lượng chất xơ (g).

carbo: Tổng lượng carbohydrate (g).

sugars: Lượng đường (g).

potass: Hàm lượng kali (mg).

vitamins: Tỷ lệ % vitamin khuyến nghị hàng ngày.

shelf: Vị trí kệ trưng bày (1 = đáy, 3 = trên cùng).

weight: Trọng lượng một khẩu phần (oz).

cups: Số cốc trong mỗi khẩu phần.

rating: Điểm đánh giá của người tiêu dùng (càng cao càng tốt).

##### (a) Với một biến định lượng, ước lượng kỳ vọng tổng thể bằng khoảng tin cậy 95%.
##### (b) Với một biến định lượng, ước lượng phương sai tổng thể bằng khoảng tin cậy 95%.
##### (c) Với một biến định tính có hai giá trị (nhị phân), ước lượng tỷ lệ tổng thể bang khoảng tin cậy 95%.
##### (d) Chọn hai nhóm khác nhau trong dữ liệu (dựa trên biến định tính), so sánh kỳ vọng của một biến định lượng giữa hai nhóm bằng cách xây dựng khoảng tin cậy cho hiệu kỳ vọng µ1 - µ2.


In [126]:
import pandas as pd
import numpy as np
import math
from scipy.stats import norm, chi2

In [127]:
data = pd.read_csv('cleaned_cereal.csv')
data = data[~data['shelf'].isin([2])]
data.head()

,name,mfr,type,calories,protein,fat,sodium,fiber,carbo,sugars,potass,vitamins,shelf,weight,cups,rating
0,100% Bran,N,C,70,4,1,130,10.0,5.0,6,280,25,3,1.0,0.33,68.402973
1,100% Natural Bran,Q,C,120,3,5,15,2.0,8.0,8,135,0,3,1.0,1.00,33.983679
2,All-Bran,K,C,70,4,1,260,9.0,7.0,5,320,25,3,1.0,0.33,59.425505
3,All-Bran with Extra Fiber,K,C,50,4,0,140,14.0,8.0,0,330,25,3,1.0,0.50,93.704912
4,Almond Delight,R,C,110,2,2,200,1.0,14.0,8,-1,25,3,1.0,0.75,34.384843


In [128]:
data.describe()

,calories,protein,fat,sodium,fiber,carbo,sugars,potass,vitamins,shelf,weight,cups,rating
count,71.000000,71.000000,71.000000,71.000000,71.000000,71.000000,71.000000,71.000000,71.000000,71.000000,71.000000,71.000000,71.000000
mean,106.760563,2.577465,1.014085,161.267606,2.207042,14.549296,6.859155,96.661972,28.521127,2.014085,1.027465,0.802394,42.858021
std,20.055256,1.091028,1.035001,82.017935,2.412605,4.372932,4.469886,71.177433,23.259320,1.007018,0.152407,0.223514,14.085830
min,50.000000,1.000000,0.000000,0.000000,0.000000,-1.000000,-1.000000,-1.000000,0.000000,1.000000,0.500000,0.250000,18.042851
25%,100.000000,2.000000,0.000000,132.500000,1.000000,12.000000,3.000000,40.000000,25.000000,1.000000,1.000000,0.670000,33.578886
50%,110.000000,3.000000,1.000000,180.000000,2.000000,14.000000,7.000000,90.000000,25.000000,3.000000,1.000000,0.750000,40.448772
75%,110.000000,3.000000,2.000000,210.000000,3.000000,17.000000,10.500000,120.000000,25.000000,3.000000,1.000000,1.000000,50.796695
max,160.000000,6.000000,5.000000,320.000000,14.000000,23.000000,15.000000,330.000000,100.000000,3.000000,1.500000,1.330000,93.704912


In [129]:
data = data[data['carbo'] >= 0]
n = len(data)

In [130]:
def mean(x):
    # Trung bình mẫu:  x̄ = (1/n) Σ x_i
    x = np.asarray(x, dtype=float)
    return x.sum() / len(x)

def var_sample(x):
    # Phương sai mẫu hiệu chỉnh:  s² = Σ (x_i - x̄)² / (n - 1)
    x = np.asarray(x, dtype=float)
    xb = mean(x)
    return ((x - xb) ** 2).sum() / (len(x) - 1)

def std_sample(x):
    # Độ lệch chuẩn
    return np.sqrt(var_sample(x))

def print_ci(name, lo, hi, unit=""):
    print(f"Khoảng tin cậy 95% cho {name}: ({lo:.4f}{unit} ; {hi:.4f}{unit})")

##### (a) Với một biến định lượng, ước lượng kỳ vọng tổng thể bằng khoảng tin cậy 95%.
+ X là biến ngẫu nhiên chỉ lượng đường trong 1 loại ngũ cốc(carbo), n = 77
+ vì n > 30, áp dụng định lí giới hạn trung tâm ta được: $\bar{X} \approx N(\mu, \sigma^2/n)$
+ chọn thống kê: $Z = \frac{\bar{X} - \mu}{S / \sqrt{n}} \sim N(0,1)$
+ KTC đối xứng: $\bar{X} \pm z_{\alpha/2} \cdot \frac{S}{\sqrt{n}}$

In [131]:
alpha = 0.05
x_ngang = mean(data['carbo'])
ps_hc = var_sample(data['carbo'])
z_alpha_2 = norm.ppf(1 - alpha/2)
print('Áp dụng mẫu cụ thể:')
print('+ 1 - alpha = 0.95 => alpha = 0.05')
print(f'+ x_ngang \u2248 {x_ngang}')
print(f'+ n = {n}')
print(f'+ s \u2248 {math.sqrt(ps_hc)}')
print_ci('Ước lượng cho lượng đường', x_ngang - z_alpha_2*math.sqrt(ps_hc/n), x_ngang + z_alpha_2*math.sqrt(ps_hc/n))

Áp dụng mẫu cụ thể:
+ 1 - alpha = 0.95 => alpha = 0.05
+ x_ngang ≈ 14.771428571428572
+ n = 70
+ s ≈ 3.980646763309238
Khoảng tin cậy 95% cho Ước lượng cho lượng đường: (13.8389 ; 15.7039)


##### (b) Với một biến định lượng, ước lượng phương sai tổng thể bằng khoảng tin cậy 95%.
+ Ta chọn thống kê: $K^2 = (n-1).S^2/\sigma^2 $~$ \chi^2_{n-1}$
+ KTC đối xứng: $\frac{(n-1)S^2}{\chi^2_{\alpha/2, n-1}} < \sigma^2 < \frac{(n-1)S^2}{\chi^2_{1-\alpha/2, n-1}}$


In [132]:
alpha = 0.05
x_ngang = mean(data['carbo'])
ps_hc = var_sample(data['carbo'])
chi_alpha_2 = chi2.ppf(1 - alpha/2, n-1)
chi_1alpha2 = chi2.ppf(alpha/2, n-1)
print('Áp dụng mẫu cụ thể:')
print('+ 1 - alpha = 0.95 => alpha = 0.05')
print(f'+ x_ngang \u2248 {x_ngang:.4f}')
print(f'+ n = {n}')
print(f'+ s \u2248 {math.sqrt(ps_hc):.4f}')
print_ci('phương sai của lượng đường', (n-1)*ps_hc/chi_alpha_2, (n-1)*ps_hc/chi_1alpha2)

Áp dụng mẫu cụ thể:
+ 1 - alpha = 0.95 => alpha = 0.05
+ x_ngang ≈ 14.7714
+ n = 70
+ s ≈ 3.9806
Khoảng tin cậy 95% cho phương sai của lượng đường: (11.6491 ; 22.8140)


##### (c) Với một biến định tính có hai giá trị (nhị phân), ước lượng tỷ lệ tổng thể bằng khoảng tin cậy 95%.
+ Gọi Y là biến ngẫu nhiên thoả mãn:$
Y = \begin{cases}
0 & \text{nếu ngũ cốc là ở dưới} \\
1 & \text{nếu ngũ cốc là ở trên}
\end{cases}
$
+ P(Y = 1) = p
+ Ta dùng $ \hat{P} = \Sigma Y_i/n$ để ước lượng cho p. $\hat{P} \sim N(p;p(1-p))$
+ Chọn thống kê $Z = \frac{\hat{P} - p}{\sqrt{\frac{p(1-p)}{n}}} \approx N(0,1)$
+ KTC đối xứng: $
\hat{p} - z_{\alpha/2}\sqrt{\frac{\hat{p}(1-\hat{p})}{n}} < p < \hat{p} + z_{\alpha/2}\sqrt{\frac{\hat{p}(1-\hat{p})}{n}}
$

In [133]:
alpha = 0.05
p_hat = len(data[data['shelf'].isin([1])]) / n
z_alpha_2 = norm.ppf(1 - alpha/2)
print('Áp dụng mẫu cụ thể:')
print('+ 1 - alpha = 0.95 => alpha = 0.05')
print(f'+ p_hat \u2248 {p_hat:.4f}')
print(f'+ n = {n}')
print_ci('tỷ lệ ngũ cốc là ở dưới', p_hat - z_alpha_2*math.sqrt(p_hat*(1-p_hat)/n), p_hat + z_alpha_2*math.sqrt(p_hat*(1-p_hat)/n))

Áp dụng mẫu cụ thể:
+ 1 - alpha = 0.95 => alpha = 0.05
+ p_hat ≈ 0.4857
+ n = 70
Khoảng tin cậy 95% cho tỷ lệ ngũ cốc là ở dưới: (0.3686 ; 0.6028)


##### (d) Chọn hai nhóm khác nhau trong dữ liệu (dựa trên biến định tính), so sánh kỳ vọng của một biến định lượng giữa hai nhóm bằng cách xây dựng khoảng tin cậy cho hiệu kỳ vọng µ1 - µ2.
+ X1 là biến ngẫu nhiên chỉ lượng đường trong 1 loại ngũ cốc ở dưới, n1 = 35
+ X2 là biến ngẫu nhiên chỉ lượng đường trong 1 loại ngũ cốc ở trên, n2 = 36
+ vì n1, n2 > 30, áp dụng định lí giới hạn trung tâm ta được: $\bar{X_i} \approx N(\mu_i, \sigma_i^2/n_i)$
+ chọn thống kê: $
Z = \frac{(\bar{X}_1 - \bar{X}_2) - (\mu_1 - \mu_2)}{\sqrt{\frac{S_1^2}{n_1} + \frac{S_2^2}{n_2}}}
$
+ KTC đối xứng: $
(\bar{x}_1 - \bar{x}_2) - z_{\alpha/2}\sqrt{\frac{S_1^2}{n_1} + \frac{S_2^2}{n_2}} < \mu_1 - \mu_2 < (\bar{x}_1 - \bar{x}_2) + z_{\alpha/2}\sqrt{\frac{S_1^2}{n_1} + \frac{S_2^2}{n_2}}
$


In [136]:
alpha = 0.05
n1 = len(data[data['shelf'].isin([1])]['carbo'])
n2 = len(data[data['shelf'].isin([3])]['carbo'])
x_1 = mean(data[data['shelf'].isin([1])]['carbo'])
x_2 = mean(data[data['shelf'].isin([3])]['carbo'])

ps_hc1 = var_sample(data[data['shelf'].isin([1])]['carbo'])
ps_hc2 = var_sample(data[data['shelf'].isin([3])]['carbo'])

z_alpha_2 = norm.ppf(1 - alpha/2)
print('Áp dụng mẫu cụ thể:')
print('+ 1 - alpha = 0.95 => alpha = 0.05')
print(f'+ x1_ngang \u2248 {x_1:.4f}')
print(f'+ x2_ngang \u2248 {x_2:.4f}')

print(f'+ n1 = {n1}')
print(f'+ n2 = {n2}')

print(f'+ s1 \u2248 {math.sqrt(ps_hc1):.4f}')
print(f'+ s2 \u2248 {math.sqrt(ps_hc2):.4f}')

print_ci('Ước lượng cho lượng đường', x_1-x_2 - z_alpha_2*math.sqrt(ps_hc1/n1+ps_hc2/n2), x_1-x_2 + z_alpha_2*math.sqrt(ps_hc1/n1+ps_hc2/n2))

Áp dụng mẫu cụ thể:
+ 1 - alpha = 0.95 => alpha = 0.05
+ x1_ngang ≈ 15.0588
+ x2_ngang ≈ 14.5000
+ n1 = 34
+ n2 = 36
+ s1 ≈ 3.7594
+ s2 ≈ 4.2139
Khoảng tin cậy 95% cho Ước lượng cho lượng đường: (-1.3098 ; 2.4274)


=>Vì 0 thuộc KTC nên chưa đủ bằng chứng cho thấy trung bình hai nhóm khác nhau.